In [1]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP
# =====================================================================
!pip install transformers accelerate datasets spacy pillow tqdm torch
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
!pip install -U bitsandbytes transformers accelerate datasets spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [3]:
# =====================================================================
# CELL 2: IMPORTS AND MODEL LOADING (4-BIT VRAM-SAFE VERSION)
# =====================================================================
import os
import json
import gc
import re
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from collections import defaultdict
from PIL import Image
import spacy

# Import 4-bit quantization tools
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, OwlViTProcessor, OwlViTForObjectDetection, BitsAndBytesConfig

nlp = spacy.load("en_core_web_sm")

# 1. Shink the model to 5GB using 4-bit Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load Qwen2.5-VL in 4-bit
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
tokenizer = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# 3. Load OWL-ViT
owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to("cuda")

print("All models loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  613MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

All models loaded successfully!


In [4]:
# =====================================================================
# CELL 3: PIPELINE FUNCTIONS
# =====================================================================
def extract_visual_claims(text):
    """Extracts short concrete nouns from the reasoning trace using NLP."""
    doc = nlp(text)
    nouns = [chunk.text.lower().strip() for chunk in doc.noun_chunks if len(chunk.text.split()) < 4]
    return list(set(nouns))[:10]

def owl_grounding_score(image, text):
    """Checks if the objects mentioned in the text actually exist in the image."""
    claims = extract_visual_claims(text)
    if not claims:
        return 1.0

    # FIX: max_length=16 prevents both jagged tensors AND exceeds OWL-ViT's token limit
    inputs = owl_processor(
        text=[claims],
        images=image,
        return_tensors="pt",
        padding="max_length",
        max_length=16,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = owl_model(**inputs)

    probs = torch.sigmoid(outputs.logits[0])
    max_confidences = probs.max(dim=0).values
    return max_confidences.mean().item()

def extract_final_answer(text):
    """Standardizes answer extraction for majority voting."""
    match = re.search(r'(?:[Tt]he answer is|answer:|choose)\s*([A-D0-9\.\-\/]+)', text)
    if match: return match.group(1).strip()
    nums = re.findall(r'-?\d*\.?\d+', text)
    return nums[-1] if nums else text[-10:].strip()

In [5]:
# =====================================================================
# CELL 4: CONFIDENCE-WEIGHTED SELF-CONSISTENCY (CISC)
# =====================================================================
def cisc_generate_and_vote(image, question, num_samples=3):
    # 1. VRAM PROTECTION: Resize huge images so attention matrices don't explode
    if max(image.size) > 768:
        ratio = 768 / max(image.size)
        image = image.resize((int(image.size[0] * ratio), int(image.size[1] * ratio)), Image.Resampling.LANCZOS)

    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], images=[image], return_tensors="pt").to("cuda")

    candidates = []
    answer_votes = defaultdict(float)

    for _ in range(num_samples):
        with torch.no_grad():
            # Temperature must be > 0 for self-consistency to generate diverse paths
            out_ids = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
            reasoning = tokenizer.decode(out_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

            conf_score = owl_grounding_score(image, reasoning)
            ans = extract_final_answer(reasoning)
            answer_votes[ans] += conf_score

            candidates.append({"reasoning": reasoning, "answer": ans, "visual_confidence": conf_score})

        # 2. VRAM PROTECTION: Empty the GPU trash after every single sample
        del out_ids
        torch.cuda.empty_cache()
        gc.collect()

    del inputs
    torch.cuda.empty_cache()

    best_answer = max(answer_votes, key=answer_votes.get)
    best_reasoning = next(c["reasoning"] for c in candidates if c["answer"] == best_answer)
    return best_answer, best_reasoning, dict(answer_votes)

In [7]:
# =====================================================================
# CELL 5: ACADEMIC BATCH PIPELINE (SYNC-SAFE VERSION)
# =====================================================================
import os, json
from tqdm.auto import tqdm
from datasets import load_dataset

# Ensure Google Drive is mounted (will prompt for authorization if not already mounted)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass # Ignore if not running in Colab

dataset = load_dataset("AI4Math/MathVista", split="testmini")

# Save file set up for Google Colab Drive
GSV_RESULTS_FILE = "/content/drive/MyDrive/Colab_Storage/gsv_math_results/cisc_owl_gsv_results.jsonl"
os.makedirs(os.path.dirname(GSV_RESULTS_FILE), exist_ok=True)

completed = set()

# 1. Safely load existing progress line-by-line
if os.path.exists(GSV_RESULTS_FILE):
    with open(GSV_RESULTS_FILE, "r") as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    completed.add(data["pid"])
                except json.JSONDecodeError:
                    pass # Ignore partial lines if a crash happened mid-write

print(f"Resuming: {len(completed)}/1000 academic evaluations complete.")

remaining_samples = [s for s in dataset if s["pid"] not in completed]

# 2. Open file in APPEND mode ("a") to prevent overwriting
with open(GSV_RESULTS_FILE, "a") as f:
    for sample in tqdm(remaining_samples):
        pid = sample["pid"]

        best_ans, best_reasoning, votes = cisc_generate_and_vote(
            sample["decoded_image"],
            sample["query"],
            num_samples=3
        )

        result_dict = {
            "pid": pid,
            "ground_truth": sample["answer"],
            "cisc_final_answer": best_ans,
            "raw_response": best_reasoning,
            "vote_distribution": votes
        }

        # 3. Write exactly one line
        f.write(json.dumps(result_dict) + "\n")

        # 4. FORCE Python to flush to the hard drive immediately!
        f.flush()
        os.fsync(f.fileno())

print("CISC Grounding Pipeline Complete!")

Mounted at /content/drive


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…): reconstructing file:   0%|          |  0.00B /  142MB            

data/testmini-00000-of-00001-725687bf7a1(…): downloading bytes:           |  0.00B            

data/test-00000-of-00002-6b81bd7f7e2065e(…): reconstructing file:   0%|          |  0.00B /  358MB            

...

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

Resuming: 912/1000 academic evaluations complete.


  0%|          | 0/88 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


CISC Grounding Pipeline Complete!
